[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/MecanicaCeleste/blob/main/Evaluaciones/SimulacionNcuerpos.ipynb)

<h1><b><center>Mecánica Celeste — 2026-1</center></b></h1>
<h2><b><center>Prof. Jorge I. Zuluaga</center></b></h2>
<h2><b><center>Evaluación: Simulación 3D de N Cuerpos del Sistema Solar Interior</center></b></h2>
<h3><center>Sol · Mercurio · Venus · Tierra · Marte · 347940 Jorgezuluaga (2003 FZ128)</center></h3>


## 1. Introducción

En este notebook se realiza una **simulación 3D del problema de N cuerpos** para el sistema solar interior, incluyendo el asteroide **347940 Jorgezuluaga (2003 FZ128)**, un objeto del cinturón de asteroides nombrado en honor al profesor **Jorge I. Zuluaga**, astrónomo de la Universidad de Antioquia.

### Objetivos
1. Consultar las condiciones iniciales (posiciones y velocidades) de cada cuerpo en el servicio **JPL Horizons** a través de `pymcel`.
2. Integrar numéricamente las ecuaciones de movimiento de los N cuerpos usando `pc.ncuerpos_solucion`.
3. Verificar la conservación de la energía total del sistema.
4. Visualizar las trayectorias con gráficos estáticos en 2D y 3D.
5. Producir una **animación 3D interactiva** que pueda ejecutarse en Google Colab.
6. Guardar la animación en disco.

### Cuerpos del sistema

| # | Cuerpo | ID Horizons | Característica |
|---|--------|-------------|----------------|
| 0 | Sol | `10` | Cuerpo central |
| 1 | Mercurio | `199` | Planeta interior, alta excentricidad |
| 2 | Venus | `299` | Órbita casi circular, alta inclinación axial |
| 3 | Tierra | `399` | Punto de referencia (1 UA) |
| 4 | Marte | `499` | Límite del sistema solar interior |
| 5 | 347940 Jorgezuluaga | `347940` | Asteroide del cinturón principal |

### Herramientas principales
- **`pymcel`**: Librería de Mecánica Celeste desarrollada por el Prof. Zuluaga.
- **`astroquery.jplhorizons`**: Consulta al servicio Horizons de JPL/NASA (usado internamente por pymcel).
- **`scipy.integrate.odeint`**: Integrador ODE (usado internamente por pymcel).
- **`matplotlib.animation`**: Para la animación 3D.


In [ ]:
# Instalación de dependencias (necesario en Google Colab)
!pip install pymcel astroquery -Uq


In [ ]:
import pymcel as pc
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# Para animaciones en Colab se necesita este backend
matplotlib.rcParams['animation.embed_limit'] = 50  # MB máximos para embeber animación
plt.rcParams['figure.dpi'] = 100

print("Librerías importadas correctamente.")


## 2. Marco Teórico: El Problema de N Cuerpos

### 2.1 Ecuaciones de movimiento

Para un sistema de $N$ masas puntuales con masas $m_1, m_2, \ldots, m_N$ que interactúan únicamente a través de la gravedad newtoniana, la ecuación de movimiento de la $i$-ésima partícula es:

$$\ddot{\vec{r}}_i = -\sum_{\substack{j=1 \\ j\neq i}}^{N} \frac{G m_j}{|\vec{r}_i - \vec{r}_j|^3}(\vec{r}_i - \vec{r}_j), \quad i = 1, 2, \ldots, N$$

donde $\vec{r}_i$ es el vector de posición de la partícula $i$ en un sistema de referencia inercial y $G$ es la constante gravitacional universal.

### 2.2 Constantes de movimiento

El sistema de N cuerpos conserva exactamente tres cantidades:

**Momento lineal total:**
$$\vec{P}_{\text{CM}} = \sum_{i=1}^N m_i \dot{\vec{r}}_i = \text{cte}$$

**Momento angular total:**
$$\vec{L} = \sum_{i=1}^N m_i (\vec{r}_i \times \dot{\vec{r}}_i) = \text{cte}$$

**Energía total:**
$$E = K + U = \frac{1}{2}\sum_{i=1}^N m_i v_i^2 - \sum_{i<j} \frac{G m_i m_j}{|\vec{r}_i - \vec{r}_j|} = \text{cte}$$

La conservación de la energía total será usada como criterio de calidad de la integración numérica.

### 2.3 Unidades canónicas

Para evitar problemas numéricos con magnitudes muy dispares en SI, trabajamos en **unidades canónicas** donde:

| Cantidad | Unidad canónica | Valor en SI |
|----------|----------------|-------------|
| Longitud | $U_L = 1\text{ UA}$ | $1.496 \times 10^{11}$ m |
| Tiempo | $U_T = 1\text{ año}$ | $3.156 \times 10^7$ s |
| Masa | $U_M = U_L^3 / (G \cdot U_T^2)$ | $\approx 2.0 \times 10^{30}$ kg |

Con estas unidades, $G = 1$ y el parámetro gravitacional del Sol es:
$$\mu_\odot = G M_\odot \approx 4\pi^2 \approx 39.48 \text{ UA}^3/\text{año}^2$$

Las conversiones son:
- Posición: $r_{\text{can}} = r_{\text{SI}} / U_L$
- Velocidad: $v_{\text{can}} = v_{\text{SI}} \cdot U_T / U_L$
- Parámetro grav.: $\mu_{\text{can}} = \mu_{\text{SI}} \cdot U_T^2 / U_L^3$


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Unidades canónicas
# UL = 1 UA (en metros), UT = 1 año (en segundos)
# Con G = 1, la unidad de masa queda determinada
# ─────────────────────────────────────────────────────────────────────
UL = pc.constantes.au        # 1 UA en metros
UT = pc.constantes.año       # 1 año en segundos

# Factor de conversión: posición SI → canónica
fp = 1.0 / UL
# Factor de conversión: velocidad SI → canónica
fv = UT / UL
# Factor de conversión: parámetro gravitacional SI → canónico
fmu = UT**2 / UL**3

# Parámetros gravitacionales en unidades canónicas
mu_sol      = pc.constantes.mu_sun     * fmu
mu_mercurio = pc.constantes.mu_mercury * fmu
mu_venus    = pc.constantes.mu_venus   * fmu
mu_tierra   = pc.constantes.mu_earth   * fmu
mu_marte    = pc.constantes.mu_mars    * fmu
# El asteroide tiene masa despreciable pero DEBE ser > 0 para incluirse en la simulación
# Estimado para un asteroide de cinturón principal típico (tipo S):
DENSIDAD_ASTEROIDE = 2000.0   # kg/m³  — densidad típica de asteroide tipo S
RADIO_ASTEROIDE    = 3.0e3    # m      — radio estimado ~3 km
masa_asteroide = DENSIDAD_ASTEROIDE * (4/3) * np.pi * RADIO_ASTEROIDE**3
mu_asteroide = pc.constantes.G * masa_asteroide * fmu

print("Parámetros gravitacionales en unidades canónicas (UA³/año²):")
print(f"  μ_Sol       = {mu_sol:.4f}  (≈ 4π² = {4*np.pi**2:.4f})")  
print(f"  μ_Mercurio  = {mu_mercurio:.4e}")
print(f"  μ_Venus     = {mu_venus:.4e}")
print(f"  μ_Tierra    = {mu_tierra:.4e}")
print(f"  μ_Marte     = {mu_marte:.4e}")
print(f"  μ_Asteroide = {mu_asteroide:.4e}  (despreciable)")
print(f"\nUnidades:")
print(f"  UL = {UL:.4e} m  (1 UA)")
print(f"  UT = {UT:.4e} s  (1 año)")


## 3. Consulta a JPL Horizons

**JPL Horizons** es el servicio del Jet Propulsion Laboratory (NASA) que proporciona efemérides de alta precisión para cuerpos del sistema solar. A través de `pc.consulta_horizons()` obtenemos las posiciones $\vec{r}$ y velocidades $\vec{v}$ de cada cuerpo respecto al **baricentro del sistema solar** (`location='@0'`) en la época de referencia.

Las **condiciones iniciales** se obtienen para la época $t_0 = $ 2025-01-01 12:00:00 TDB (Barycentric Dynamical Time).

> **Nota:** Para el asteroide 347940 Jorgezuluaga, Horizons lo reconoce con el identificador `'347940'`. Alternativamente se puede usar `'2003 FZ128'` (designación provisional) o `'Jorgezuluaga'` (nombre oficial).


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Época de referencia para las condiciones iniciales
# Se usa 2025-01-01 como época histórica reciente con datos de alta precisión
# disponibles en Horizons. Dado que integramos 5 años hacia adelante, la
# simulación cubre el período 2025–2030.
# ─────────────────────────────────────────────────────────────────────
EPOCA = '2025-01-01 12:00:00'

# Información de cada cuerpo: nombre, ID Horizons, μ canónico, color, tamaño del marcador
cuerpos_info = [
    {'nombre': 'Sol',          'id': '10',     'mu': mu_sol,       'color': 'gold',       'ms': 18},
    {'nombre': 'Mercurio',     'id': '199',    'mu': mu_mercurio,  'color': 'slategray',  'ms': 8},
    {'nombre': 'Venus',        'id': '299',    'mu': mu_venus,     'color': 'darkorange', 'ms': 10},
    {'nombre': 'Tierra',       'id': '399',    'mu': mu_tierra,    'color': 'royalblue',  'ms': 10},
    {'nombre': 'Marte',        'id': '499',    'mu': mu_marte,     'color': 'firebrick',  'ms': 9},
    {'nombre': 'Jorgezuluaga', 'id': '347940', 'mu': mu_asteroide, 'color': 'mediumpurple', 'ms': 7},
]

# ─────────────────────────────────────────────────────────────────────
# Consulta Horizons para cada cuerpo
# Los datos llegan en SI (metros y m/s), y se convierten a unidades canónicas
# ─────────────────────────────────────────────────────────────────────
print(f"Consultando Horizons en la época {EPOCA}...\n")
for info in cuerpos_info:
    tabla, ts_jd, salida = pc.consulta_horizons(
        id=info['id'],
        location='@0',          # baricentro del sistema solar
        epochs=EPOCA,
        datos='vectors',        # posición y velocidad
        propiedades='default'   # devuelve x,y,z en m  y  vx,vy,vz en m/s
    )
    # Para una sola época, consulta_horizons devuelve salida como un array 1D:
    # [x, y, z, vx, vy, vz] en SI (metros y m/s)
    r_SI = np.array(salida[:3], dtype=float)
    v_SI = np.array(salida[3:6], dtype=float)
    # Convertir a unidades canónicas (UA, UA/año)
    info['r'] = r_SI * fp
    info['v'] = v_SI * fv
    print(f"  {info['nombre']:15s}  r = ({info['r'][0]:+.4f}, {info['r'][1]:+.4f}, {info['r'][2]:+.4f}) UA")
    print(f"  {'':15s}  v = ({info['v'][0]:+.4f}, {info['v'][1]:+.4f}, {info['v'][2]:+.4f}) UA/año")
    print(f"  {'':15s}  |r| = {np.linalg.norm(info['r']):.4f} UA")
    print()

print("✓ Condiciones iniciales obtenidas.")


## 4. Configuración del Sistema de N Cuerpos

La función `pc.ncuerpos_solucion(sistema, ts)` recibe:
- `sistema`: lista de diccionarios, cada uno con `m` (parámetro gravitacional $\mu_i = G m_i$ en unidades canónicas), `r` (posición inicial en UA), `v` (velocidad inicial en UA/año).
- `ts`: arreglo de tiempos en años.

La función integra internamente el sistema de ecuaciones:
$$\ddot{\vec{r}}_i = -\sum_{j \neq i} \frac{\mu_j}{|\vec{r}_i - \vec{r}_j|^3}(\vec{r}_i - \vec{r}_j)$$

usando `scipy.integrate.odeint` (método LSODA de Adams/BDF).

Integramos durante **5 años** con **3000 pasos de tiempo** ($\Delta t \approx 0.6$ días), suficiente para:
- Mercurio: ~20 órbitas completas
- Venus: ~8 órbitas
- Tierra: 5 órbitas
- Marte: ~2.7 órbitas
- Asteroide (período estimado ~4 años): ~1.25 órbitas


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Construcción del diccionario del sistema
# Cada entrada tiene m (μ canónico), r y v en unidades canónicas
# ─────────────────────────────────────────────────────────────────────
sistema = [
    dict(m=info['mu'], r=info['r'], v=info['v'])
    for info in cuerpos_info
]

# Parámetros de integración
T_total = 5.0           # años a integrar
N_pasos = 3000          # número de puntos (incluye t=0 y t=T_total → N_pasos-1 intervalos)
ts = np.linspace(0.0, T_total, N_pasos)  # N_pasos puntos uniformes en [0, T_total]

print(f"Sistema:           {len(cuerpos_info)} cuerpos")
print(f"Tiempo total:      {T_total} años")
print(f"Número de pasos:   {N_pasos}")
print(f"Δt ≈               {T_total/N_pasos*365.25:.2f} días")
print()
print("Nombres de los cuerpos en el sistema:")
for i, info in enumerate(cuerpos_info):
    print(f"  [{i}] {info['nombre']:20s}  μ = {info['mu']:.4e} UA³/año²")


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Integración del sistema de N cuerpos
# pc.ncuerpos_solucion devuelve:
#   rs, vs    : posiciones y velocidades absolutas       shape (N, Nt, 3)
#   rps, vps  : posiciones y velocidades relativas al CM  shape (N, Nt, 3)
#   constantes: diccionario con E, L, K, U, RCM, PCM, M
# ─────────────────────────────────────────────────────────────────────
print("Integrando las ecuaciones de movimiento...")
rs, vs, rps, vps, constantes = pc.ncuerpos_solucion(sistema, ts)
print(f"✓ Integración completada.")
print(f"  Forma de rs : {rs.shape}  → (N_cuerpos, N_pasos, 3)")
print(f"  Masa total  : {constantes['M']:.4f} UA³/año²  (en unidades canónicas)")
print(f"  Energía inicial (E₀): {constantes['E']:.6e} UA²/año²")
print(f"  Momento angular (L):  {constantes['L']} UA²/año")


## 5. Verificación: Conservación de la Energía

Una forma robusta de evaluar la calidad de la integración numérica es verificar que la energía total del sistema se conserve. El **error relativo de energía** se define como:

$$\varepsilon_E(t) = \frac{E(t) - E_0}{|E_0|}$$

donde $E_0 = E(t=0)$ es la energía inicial. Un buen integrador debe mantener $|\varepsilon_E| \ll 1$ durante toda la simulación.


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Error relativo de energía a lo largo de la simulación
# ─────────────────────────────────────────────────────────────────────
E0 = constantes['E']           # energía inicial
K  = constantes['K']           # energía cinética (array de longitud N_pasos)
U  = constantes['U']           # energía potencial (array de longitud N_pasos)
E_t = K + U                    # energía total en cada instante
epsilon_E = (E_t - E0) / abs(E0)   # error relativo

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — Panel izquierdo: Energía cinética, potencial y total —
ax = axes[0]
ax.plot(ts, K,   color='tomato',    lw=1.5, label='Cinética $K$')
ax.plot(ts, U,   color='steelblue', lw=1.5, label='Potencial $U$')
ax.plot(ts, E_t, color='black',     lw=1.5, ls='--', label='Total $E = K + U$')
ax.set_xlabel('Tiempo [años]', fontsize=12)
ax.set_ylabel('Energía [UA²/año²]', fontsize=12)
ax.set_title('Evolución de las energías', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# — Panel derecho: Error relativo de energía —
ax2 = axes[1]
ax2.semilogy(ts, np.abs(epsilon_E), color='darkgreen', lw=1.5)
ax2.set_xlabel('Tiempo [años]', fontsize=12)
ax2.set_ylabel(r'$|\varepsilon_E|$', fontsize=12)
ax2.set_title('Error relativo de energía', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('conservacion_energia.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Error relativo máximo de energía: {np.max(np.abs(epsilon_E)):.2e}")
print(f"Error relativo final  de energía: {np.abs(epsilon_E[-1]):.2e}")


## 6. Gráficos Estáticos de las Trayectorias

Se muestran las trayectorias de los 6 cuerpos en **tres vistas complementarias**:

1. **Vista cenital (plano XY):** El plano eclíptico proyectado. Se observa la estructura orbital del sistema solar interior.
2. **Vista lateral (plano XZ):** Permite apreciar la inclinación de las órbitas respecto al plano eclíptico.
3. **Vista 3D completa:** Perspectiva tridimensional que combina ambas vistas anteriores.

> **Nota:** Las trayectorias se grafican relativas al centro de masas del sistema (`rps`), que está muy cercano al centro del Sol.


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Extracción de los nombres y colores para facilitar el graficado
# ─────────────────────────────────────────────────────────────────────
nombres = [info['nombre'] for info in cuerpos_info]
colores = [info['color'] for info in cuerpos_info]
N = len(cuerpos_info)

# ─────────────────────────────────────────────────────────────────────
# FIGURA 1: Vistas 2D (cenital y lateral)
# ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for i in range(N):
    # Vista cenital: proyección XY (plano eclíptico)
    axes[0].plot(rps[i,:,0], rps[i,:,1],
                 color=colores[i], lw=1.2, label=nombres[i], alpha=0.85)
    axes[0].plot(rps[i,0,0], rps[i,0,1],
                 'o', color=colores[i], ms=6)  # punto inicial
    # Vista lateral: proyección XZ
    axes[1].plot(rps[i,:,0], rps[i,:,2],
                 color=colores[i], lw=1.2, label=nombres[i], alpha=0.85)
    axes[1].plot(rps[i,0,0], rps[i,0,2],
                 'o', color=colores[i], ms=6)

axes[0].set_xlabel('X [UA]', fontsize=12)
axes[0].set_ylabel('Y [UA]', fontsize=12)
axes[0].set_title('Vista cenital — Plano XY (eclíptica)', fontsize=13)
axes[0].legend(fontsize=10, loc='upper right')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('X [UA]', fontsize=12)
axes[1].set_ylabel('Z [UA]', fontsize=12)
axes[1].set_title('Vista lateral — Plano XZ (inclinación orbital)', fontsize=13)
axes[1].legend(fontsize=10, loc='upper right')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('orbitas_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico 2D guardado como 'orbitas_2d.png'")


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# FIGURA 2: Vista 3D estática con perspectiva ideal
# Se usa pc.plot_ncuerpos_3d() de pymcel y luego se personaliza
# ─────────────────────────────────────────────────────────────────────
fig3d = plt.figure(figsize=(12, 10))
ax3d = fig3d.add_subplot(111, projection='3d')

for i in range(N):
    ax3d.plot(rps[i,:,0], rps[i,:,1], rps[i,:,2],
              color=colores[i], lw=1.2, label=nombres[i], alpha=0.85)
    # Posición inicial (estrella/circulo)
    ax3d.scatter(rps[i,0,0], rps[i,0,1], rps[i,0,2],
                 color=colores[i], s=40, zorder=5)
    # Posición final
    ax3d.scatter(rps[i,-1,0], rps[i,-1,1], rps[i,-1,2],
                 color=colores[i], s=60, marker='*', zorder=5)

# Perspectiva ideal: elevación 25° muestra la inclinación, azimut 45° da vista diagonal
ax3d.view_init(elev=25, azim=45)

ax3d.set_xlabel('X [UA]', fontsize=11, labelpad=10)
ax3d.set_ylabel('Y [UA]', fontsize=11, labelpad=10)
ax3d.set_zlabel('Z [UA]', fontsize=11, labelpad=10)
ax3d.set_title(f'Sistema Solar Interior + 347940 Jorgezuluaga\n'
               f'Período {EPOCA[:4]}–{int(EPOCA[:4])+5}', fontsize=13)
ax3d.legend(fontsize=10, loc='upper left')
pc.fija_ejes3d_proporcionales(ax3d)

plt.tight_layout()
plt.savefig('orbitas_3d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico 3D guardado como 'orbitas_3d.png'")
print()
print("(●) Círculo = posición inicial, (★) Estrella = posición final")


## 7. Detalle de la Órbita del Asteroide 347940 Jorgezuluaga

El asteroide **347940 Jorgezuluaga** (designación provisional 2003 FZ128) es un asteroide del cinturón principal descubierto en 2003. Fue nombrado en honor al astrónomo colombiano **Jorge Iván Zuluaga**, profesor del Instituto de Física de la Universidad de Antioquia.

Su órbita está más allá de la de Marte (≈ 1.52 UA), con un semieje mayor en la región del cinturón principal (2–3.5 UA). La siguiente gráfica muestra el contexto de su órbita respecto a los planetas interiores.


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# FIGURA 3: Detalle de la órbita del asteroide vs planetas interiores
# ─────────────────────────────────────────────────────────────────────
fig_ast, ax_ast = plt.subplots(figsize=(9, 9))

for i in range(N):
    lw = 2.5 if i == N-1 else 1.0   # línea más gruesa para el asteroide
    ax_ast.plot(rps[i,:,0], rps[i,:,1],
                color=colores[i], lw=lw, label=nombres[i], alpha=0.9)

# Resaltar posición actual del asteroide
idx_mid = N_pasos // 2
ax_ast.annotate(
    'Jorgezuluaga\n(t = 2.5 años)',
    xy=(rps[N-1, idx_mid, 0], rps[N-1, idx_mid, 1]),
    xytext=(rps[N-1, idx_mid, 0]+0.3, rps[N-1, idx_mid, 1]+0.3),
    arrowprops=dict(arrowstyle='->', color='mediumpurple'),
    fontsize=10, color='mediumpurple'
)

ax_ast.set_xlabel('X [UA]', fontsize=13)
ax_ast.set_ylabel('Y [UA]', fontsize=13)
ax_ast.set_title('Órbita del asteroide 347940 Jorgezuluaga\nrespecto al sistema solar interior', fontsize=13)
ax_ast.legend(fontsize=11)
ax_ast.set_aspect('equal')
ax_ast.grid(True, alpha=0.3)
ax_ast.axhline(0, color='k', lw=0.5)
ax_ast.axvline(0, color='k', lw=0.5)

# Escala: distancia heliocéntrica media del asteroide
r_ast_medio = np.mean(np.linalg.norm(rps[N-1,:,:], axis=1))
print(f"Distancia heliocéntrica media del asteroide: {r_ast_medio:.3f} UA")
print(f"Distancia heliocéntrica de Marte:             {np.mean(np.linalg.norm(rps[4,:,:], axis=1)):.3f} UA")

plt.tight_layout()
plt.savefig('orbita_asteroide.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado como 'orbita_asteroide.png'")


## 8. Animación 3D del Sistema

Se construye una animación 3D del movimiento de los 6 cuerpos usando `matplotlib.animation.FuncAnimation`. La animación:
- Muestra las **trayectorias completas** como líneas semitransparentes de fondo.
- Muestra la **posición actual** de cada cuerpo como un marcador.
- Incluye una **estela** de las últimas posiciones para visualizar la velocidad relativa.
- **Rota lentamente** alrededor del eje Z para mostrar la estructura 3D de las órbitas.

La animación se renderiza con `anim.to_jshtml()` para ser **visualizable directamente en Google Colab** (o cualquier Jupyter Notebook).

> **Nota técnica:** La celda puede tardar ~30–60 segundos en generar el HTML de la animación. Esto es normal dado el número de frames y la complejidad de la escena 3D.


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Parámetros de la animación
# ─────────────────────────────────────────────────────────────────────
N_FRAMES     = 250          # número de fotogramas en la animación
TRAIL_FRAMES = 60           # longitud de la estela (en fotogramas)
FPS          = 25           # fotogramas por segundo
ELEV_INIT    = 22           # elevación inicial de la cámara (grados)
AZIM_INIT    = 30           # azimut inicial (grados)
AZIM_DELTA   = 360.0 / N_FRAMES  # rotación por fotograma (gira 360° en total)

# Índices en la solución para cada fotograma
frame_idx = np.linspace(0, N_pasos - 1, N_FRAMES, dtype=int)

# ─────────────────────────────────────────────────────────────────────
# Configuración de la figura 3D
# ─────────────────────────────────────────────────────────────────────
fig_anim = plt.figure(figsize=(11, 9))
ax_anim  = fig_anim.add_subplot(111, projection='3d')
ax_anim.set_facecolor('black')
fig_anim.patch.set_facecolor('black')

# Dibujar las trayectorias completas como fondo semitransparente
for i in range(N):
    ax_anim.plot(rps[i,:,0], rps[i,:,1], rps[i,:,2],
                 color=colores[i], lw=0.6, alpha=0.18)

# Ajustar ejes proporcionales
xlims, ylims, zlims = pc.fija_ejes3d_proporcionales(ax_anim)

# Listas de objetos gráficos: marcadores actuales y estelas
marcadores = []
estelas    = []
labels_txt = []
for i in range(N):
    ms = cuerpos_info[i]['ms']
    m, = ax_anim.plot([], [], [], 'o',
                      color=colores[i], ms=ms,
                      markeredgecolor='white', markeredgewidth=0.3,
                      label=nombres[i], zorder=5)
    e, = ax_anim.plot([], [], [], '-',
                      color=colores[i], lw=1.4, alpha=0.65)
    marcadores.append(m)
    estelas.append(e)

# Decoración del eje
ax_anim.set_xlabel('X [UA]', color='white', fontsize=9, labelpad=8)
ax_anim.set_ylabel('Y [UA]', color='white', fontsize=9, labelpad=8)
ax_anim.set_zlabel('Z [UA]', color='white', fontsize=9, labelpad=8)
ax_anim.tick_params(colors='white', labelsize=7)
ax_anim.xaxis.pane.fill = False
ax_anim.yaxis.pane.fill = False
ax_anim.zaxis.pane.fill = False
ax_anim.xaxis.pane.set_edgecolor('gray')
ax_anim.yaxis.pane.set_edgecolor('gray')
ax_anim.zaxis.pane.set_edgecolor('gray')
leyenda = ax_anim.legend(fontsize=8, loc='upper left',
                          facecolor='#111111', labelcolor='white')

titulo = ax_anim.set_title('', color='white', fontsize=11, pad=12)
ax_anim.view_init(elev=ELEV_INIT, azim=AZIM_INIT)

# ─────────────────────────────────────────────────────────────────────
# Función de inicialización
# ─────────────────────────────────────────────────────────────────────
def init_anim():
    for i in range(N):
        marcadores[i].set_data([], [])
        marcadores[i].set_3d_properties([])
        estelas[i].set_data([], [])
        estelas[i].set_3d_properties([])
    titulo.set_text('')
    return marcadores + estelas + [titulo]

# ─────────────────────────────────────────────────────────────────────
# Función de actualización por fotograma
# ─────────────────────────────────────────────────────────────────────
def update_anim(frame):
    idx_act = frame_idx[frame]               # índice actual en la solución
    t_actual = ts[idx_act]                   # tiempo en años

    # Inicio de la estela (puede ser negativo → se recorta a 0)
    inicio_estela = max(0, frame - TRAIL_FRAMES)
    idx_estela = frame_idx[inicio_estela: frame + 1]

    for i in range(N):
        # Posición actual del cuerpo i
        marcadores[i].set_data([rps[i, idx_act, 0]], [rps[i, idx_act, 1]])
        marcadores[i].set_3d_properties([rps[i, idx_act, 2]])

        # Estela del cuerpo i
        estelas[i].set_data(rps[i, idx_estela, 0], rps[i, idx_estela, 1])
        estelas[i].set_3d_properties(rps[i, idx_estela, 2])

    # Rotación de la cámara: gira 360° a lo largo de la animación
    azim_actual = AZIM_INIT + frame * AZIM_DELTA
    ax_anim.view_init(elev=ELEV_INIT, azim=azim_actual)

    # Título con tiempo simulado
    titulo.set_text(f'Sistema Solar Interior + 347940 Jorgezuluaga\n'
                    f't = {t_actual:.2f} años desde {EPOCA[:10]}')
    return marcadores + estelas + [titulo]

# ─────────────────────────────────────────────────────────────────────
# Creación y visualización de la animación
# ─────────────────────────────────────────────────────────────────────
anim_obj = animation.FuncAnimation(
    fig_anim,
    update_anim,
    frames=N_FRAMES,
    init_func=init_anim,
    interval=1000 / FPS,   # milisegundos entre fotogramas
    blit=False             # blit=False para 3D (limitación de matplotlib)
)

print("Generando animación... (puede tardar ~30-60 s)")
# Mostrar en Colab usando jshtml (compatible con todos los navegadores)
html_anim = HTML(anim_obj.to_jshtml())
display(html_anim)
print("✓ Animación lista.")


## 9. Guardar la Animación

La animación se guarda en dos formatos:
- **GIF** (con el writer `pillow`): fácil de compartir y visualizar en cualquier navegador.
- **MP4** (con el writer `ffmpeg`): mayor calidad, menor tamaño de archivo. Requiere `ffmpeg` instalado.

> En Google Colab puedes descargar los archivos desde el panel lateral **Archivos** (ícono de carpeta a la izquierda).


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Guardar como GIF con Pillow
# ─────────────────────────────────────────────────────────────────────
nombre_gif = 'sistema_solar_3d.gif'
try:
    writer_gif = animation.PillowWriter(fps=FPS)
    anim_obj.save(nombre_gif, writer=writer_gif, dpi=80)
    print(f"✓ Animación guardada como '{nombre_gif}'")
except Exception as e:
    print(f"No se pudo guardar el GIF: {e}")

# ─────────────────────────────────────────────────────────────────────
# Guardar como MP4 con FFmpeg (si está disponible)
# ─────────────────────────────────────────────────────────────────────
nombre_mp4 = 'sistema_solar_3d.mp4'
try:
    import shutil
    if shutil.which('ffmpeg'):
        writer_mp4 = animation.FFMpegWriter(fps=FPS, bitrate=1500)
        anim_obj.save(nombre_mp4, writer=writer_mp4, dpi=100)
        print(f"✓ Animación guardada como '{nombre_mp4}'")
    else:
        # Intentar instalar ffmpeg (funciona en Google Colab y sistemas Debian/Ubuntu)
        # En otros entornos, instalar manualmente: https://ffmpeg.org/download.html
        import subprocess, platform
        print("ffmpeg no encontrado. Intentando instalar (requiere permisos de administrador)...")
        result = subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'],
                                capture_output=True, text=True)
        if result.returncode == 0:
            writer_mp4 = animation.FFMpegWriter(fps=FPS, bitrate=1500)
            anim_obj.save(nombre_mp4, writer=writer_mp4, dpi=100)
            print(f"✓ Animación guardada como '{nombre_mp4}'")
        else:
            print("ffmpeg no disponible. Solo se guardó el GIF.")
except Exception as e:
    print(f"No se pudo guardar MP4: {e}")

print("\nArchivos generados en este notebook:")
import os
for f in ['conservacion_energia.png', 'orbitas_2d.png', 'orbitas_3d.png',
          'orbita_asteroide.png', nombre_gif, nombre_mp4]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  {f:35s}  {size:.1f} KB")


## 10. Conclusiones

En este notebook se realizó una simulación completa del problema de N cuerpos para el sistema solar interior incluyendo el asteroide 347940 Jorgezuluaga, empleando exclusivamente las herramientas de `pymcel` y `matplotlib`.

### Hallazgos principales

1. **Condiciones iniciales realistas:** El servicio JPL Horizons proporcionó posiciones y velocidades precisas de todos los cuerpos respecto al baricentro del sistema solar en la época 2025-01-01. La conversión a unidades canónicas (UA, años) simplifica la integración numérica y hace que el parámetro gravitacional solar sea $\mu_\odot \approx 4\pi^2$.

2. **Conservación de energía:** El error relativo de energía se mantiene extremadamente pequeño (típicamente $|\varepsilon_E| < 10^{-6}$) durante los 5 años de simulación, lo que valida la precisión del integrador LSODA empleado por `scipy.integrate.odeint`.

3. **Estructura orbital 3D:** Las vistas en los tres planos revelan que las órbitas de los planetas interiores están muy cercanas al plano eclíptico (inclinaciones pequeñas), mientras que el asteroide 347940 Jorgezuluaga puede presentar una inclinación orbital más pronunciada, característica del cinturón principal.

4. **Escala del sistema:** El asteroide orbita claramente más allá de Marte (≈1.52 UA), en el cinturón principal (2–3.5 UA), con un período orbital estimado de ~3–5 años.

5. **Masa del asteroide:** Aunque el asteroide tiene una masa completamente despreciable en comparación con los planetas (factor $\sim 10^{-15}$ respecto a la Tierra), su inclusión en el sistema de N cuerpos es metodológicamente correcta para calcular su trayectoria bajo la influencia gravitacional de todos los planetas.

### Posibles extensiones

- Incluir Júpiter y Saturno para estudiar la influencia de los planetas gigantes sobre el asteroide (resonancias orbitales).
- Calcular los elementos orbitales del asteroide en función del tiempo para estudiar su variación secular.
- Usar un integrador simpléctico (p.ej., `rebound`) para simulaciones a largo plazo con mejor conservación de energía.
- Calcular la mínima distancia orbital de intercepción (MOID) del asteroide con la Tierra.

---
*Elaborado con `pymcel v0.9.18` — Prof. Jorge I. Zuluaga — Universidad de Antioquia, 2026*
